In [27]:
import pandas as pd
import numpy as np
from librosa.util import find_files
import librosa
import glob
import os
import pickle
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
import torch.nn.functional as F
from dataset import ASVspoof2019
from evaluate_tDCF_asvspoof19 import compute_eer_and_tdcf
from collections import defaultdict
from tqdm import tqdm
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import eval_metrics as em
from evaluate_tDCF_asvspoof19 import compute_eer_and_tdcf
import fnmatch

In [28]:
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [34]:
folder = "/dataHDD/neil/LOKVOX VOICE AUDIO SAMPLES/"
files = []

for root, dirnames, filenames in os.walk(folder):
    for filename in fnmatch.filter(filenames, '*.wav'):
        files.append(os.path.join(root, filename))
    for filename in fnmatch.filter(filenames, '*.mp3'):
        files.append(os.path.join(root, filename))

In [35]:
files

['/dataHDD/neil/LOKVOX VOICE AUDIO SAMPLES/3. ALERT/TIM/WEEK 10 CBSSN BUMP #2.wav',
 '/dataHDD/neil/LOKVOX VOICE AUDIO SAMPLES/3. ALERT/TIM/WEEK 10 CBS BUMP #2.wav',
 '/dataHDD/neil/LOKVOX VOICE AUDIO SAMPLES/3. ALERT/TIM/WEEK 10 TEASE.wav',
 '/dataHDD/neil/LOKVOX VOICE AUDIO SAMPLES/3. ALERT/CARIN/Indivior_Ifyoudecide_Pickup.wav',
 '/dataHDD/neil/LOKVOX VOICE AUDIO SAMPLES/3. ALERT/CARIN/USUO_DVMF_2023.wav',
 '/dataHDD/neil/LOKVOX VOICE AUDIO SAMPLES/3. ALERT/CARIN/Indivior_Opening_Pickup.wav',
 '/dataHDD/neil/LOKVOX VOICE AUDIO SAMPLES/3. ALERT/CARIN/StrategyVideos_VOG.wav',
 '/dataHDD/neil/LOKVOX VOICE AUDIO SAMPLES/3. ALERT/CARIN/StrategyVideos_Videos1-5.wav',
 '/dataHDD/neil/LOKVOX VOICE AUDIO SAMPLES/3. ALERT/CARIN/CarinGilfry_OnStar.mp3',
 '/dataHDD/neil/LOKVOX VOICE AUDIO SAMPLES/3. ALERT/CARIN/CarinGilfry_Shoprite.mp3',
 '/dataHDD/neil/LOKVOX VOICE AUDIO SAMPLES/3. ALERT/CARIN/CarinGilfry_3DProductDemo.mp3',
 '/dataHDD/neil/LOKVOX VOICE AUDIO SAMPLES/3. ALERT/CARIN/CarinGilfry

In [17]:
def test_model(feat_model_path, loss_model_path, add_loss):
    dirname = os.path.dirname
    basename = os.path.splitext(os.path.basename(feat_model_path))[0]
    if "checkpoint" in dirname(feat_model_path):
        dir_path = dirname(dirname(feat_model_path))
    else:
        dir_path = dirname(feat_model_path)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = torch.load(feat_model_path)
    # model = torch.nn.DataParallel(model, list(range(torch.cuda.device_count())))  # for multiple GPUs
    loss_model = torch.load(loss_model_path) if add_loss is not None else None
    model.eval()

[]